In [ ]:

# ================================================================
# SupplyChainAI — 03 STOCKOUT PREDICTION
# Single-Cell Jupyter Notebook
# ================================================================
#
# Features:
#  1. Load sales, inventory, products, suppliers
#  2. Validate data quality and schemas
#  3. Build SKU-level historical demand features
#  4. Estimate demand variability
#  5. Calculate inventory coverage
#  6. Calculate lead-time demand
#  7. Estimate safety stock and reorder points
#  8. Create historical stockout-risk proxy labels
#  9. Train Random Forest classifier
# 10. Evaluate with chronological holdout
# 11. Estimate current SKU/warehouse stockout risk
# 12. Generate replenishment recommendations
# 13. Visualize inventory and stockout risks
# 14. Export model, reports and predictions
#
# Install:
# pip install pandas numpy matplotlib scikit-learn joblib
#
# IMPORTANT:
# Historical stockout labels require historical inventory
# snapshots and fulfillment/lost-sales records.
#
# When these are unavailable, this notebook trains on
# simulated inventory trajectories and labels.
# Such labels are NOT verified real stockout outcomes.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

warnings.filterwarnings("ignore", category=FutureWarning)

# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42
rng = np.random.default_rng(SEED)

FORECAST_DAYS = 30
SERVICE_LEVEL_Z = 1.65
MIN_TRAIN_DAYS = 60
TEST_DAYS = 30
RISK_THRESHOLD = 0.50

ROOT = (
    Path("..")
    if Path("../data").exists() and not Path("data").exists()
    else Path(".")
)

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = ROOT / "reports" / "stockout_prediction"
MODEL_DIR = ROOT / "models" / "stockout_prediction"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("SUPPLYCHAINAI — STOCKOUT PREDICTION")
print("=" * 75)

# ================================================================
# 2. LOAD DATA
# ================================================================

def load_csv(candidates, name):
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            print(f"Loaded {name}: {path} ({len(df):,} rows)")
            return df

    raise FileNotFoundError(
        f"{name} dataset not found. Expected one of: "
        + ", ".join(str(p) for p in candidates)
    )


sales = load_csv(
    [
        PROCESSED_DIR / "sales_clean.csv",
        DATA_DIR / "sales.csv"
    ],
    "sales"
)

inventory = load_csv(
    [
        PROCESSED_DIR / "inventory_analysis.csv",
        DATA_DIR / "inventory.csv"
    ],
    "inventory"
)

products = load_csv(
    [DATA_DIR / "products.csv"],
    "products"
)

suppliers = load_csv(
    [
        PROCESSED_DIR / "supplier_analysis.csv",
        DATA_DIR / "suppliers.csv"
    ],
    "suppliers"
)

forecast_path = PROCESSED_DIR / "demand_forecast_30d.csv"

if forecast_path.exists():
    aggregate_forecast = pd.read_csv(forecast_path)
    print(f"Loaded optional aggregate forecast: {forecast_path}")
else:
    aggregate_forecast = None
    print("Optional aggregate demand forecast not found.")

# ================================================================
# 3. VALIDATION
# ================================================================

def require_columns(df, required, name):
    missing = sorted(set(required) - set(df.columns))

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )


require_columns(
    sales,
    ["date", "product_id", "quantity_sold"],
    "sales"
)

require_columns(
    inventory,
    [
        "product_id",
        "warehouse_id",
        "stock_on_hand",
        "reorder_point",
        "safety_stock"
    ],
    "inventory"
)

require_columns(
    products,
    ["product_id", "supplier_id", "unit_price"],
    "products"
)

require_columns(
    suppliers,
    ["supplier_id", "lead_time_days"],
    "suppliers"
)

sales["date"] = pd.to_datetime(
    sales["date"], errors="coerce"
).dt.normalize()

for column in ["quantity_sold"]:
    sales[column] = pd.to_numeric(
        sales[column], errors="coerce"
    )

for column in [
    "stock_on_hand",
    "reorder_point",
    "safety_stock"
]:
    inventory[column] = pd.to_numeric(
        inventory[column], errors="coerce"
    )

suppliers["lead_time_days"] = pd.to_numeric(
    suppliers["lead_time_days"], errors="coerce"
)

products["unit_price"] = pd.to_numeric(
    products["unit_price"], errors="coerce"
)

sales = sales.dropna(
    subset=["date", "product_id", "quantity_sold"]
)

sales = sales[
    sales["quantity_sold"] >= 0
].copy()

inventory = inventory.dropna(
    subset=[
        "product_id",
        "warehouse_id",
        "stock_on_hand"
    ]
)

inventory = inventory[
    inventory["stock_on_hand"] >= 0
].copy()

products = products.dropna(
    subset=["product_id", "supplier_id"]
).drop_duplicates("product_id")

suppliers = suppliers.dropna(
    subset=["supplier_id", "lead_time_days"]
).drop_duplicates("supplier_id")

if sales.empty or inventory.empty:
    raise ValueError("Sales or inventory data is empty.")

if sales["date"].nunique() < MIN_TRAIN_DAYS + TEST_DAYS:
    raise ValueError(
        "At least 90 distinct days of sales history are required."
    )

if inventory.duplicated(
    ["product_id", "warehouse_id"]
).any():
    raise ValueError(
        "Inventory must contain one current row per "
        "product_id and warehouse_id."
    )

# ================================================================
# 4. PREPARE PRODUCT AND SUPPLIER DATA
# ================================================================

supplier_columns = [
    "supplier_id",
    "lead_time_days"
]

if "on_time_delivery_rate" in suppliers.columns:
    supplier_columns.append("on_time_delivery_rate")

if "defect_rate" in suppliers.columns:
    supplier_columns.append("defect_rate")

product_supplier = products[
    ["product_id", "supplier_id", "unit_price"]
].merge(
    suppliers[supplier_columns],
    on="supplier_id",
    how="left",
    validate="many_to_one"
)

product_supplier["lead_time_days"] = (
    product_supplier["lead_time_days"]
    .fillna(7)
    .clip(lower=1)
)

if "on_time_delivery_rate" not in product_supplier:
    product_supplier["on_time_delivery_rate"] = 0.90

product_supplier["on_time_delivery_rate"] = (
    pd.to_numeric(
        product_supplier["on_time_delivery_rate"],
        errors="coerce"
    )
    .fillna(0.90)
    .clip(0, 1)
)

if "defect_rate" not in product_supplier:
    product_supplier["defect_rate"] = 0.02

product_supplier["defect_rate"] = (
    pd.to_numeric(
        product_supplier["defect_rate"],
        errors="coerce"
    )
    .fillna(0.02)
    .clip(0, 1)
)

# ================================================================
# 5. DAILY SKU/WAREHOUSE SALES PANEL
# ================================================================

# Some datasets record one warehouse per transaction.
# Build a full daily panel for observed SKU/warehouse pairs.
#
# Missing dates are assumed to mean zero recorded sales.
# This assumption must be verified for real business data.

if "warehouse_id" not in sales.columns:
    raise ValueError(
        "sales.csv needs warehouse_id for warehouse-level "
        "stockout prediction."
    )

sales_daily = (
    sales.groupby(
        ["product_id", "warehouse_id", "date"],
        as_index=False
    )
    .agg(demand=("quantity_sold", "sum"))
)

start_date = sales_daily["date"].min()
end_date = sales_daily["date"].max()

dates = pd.date_range(
    start_date,
    end_date,
    freq="D"
)

inventory_pairs = inventory[
    ["product_id", "warehouse_id"]
].drop_duplicates()

# Include inventory SKUs even when no sales were recorded.
panel = inventory_pairs.merge(
    pd.DataFrame({"date": dates}),
    how="cross"
)

panel = panel.merge(
    sales_daily,
    on=["product_id", "warehouse_id", "date"],
    how="left"
)

panel["demand"] = (
    panel["demand"]
    .fillna(0)
    .clip(lower=0)
    .astype(float)
)

panel = panel.sort_values(
    ["product_id", "warehouse_id", "date"]
).reset_index(drop=True)

print(f"\nDaily panel observations: {len(panel):,}")
print(f"SKU/warehouse pairs: {len(inventory_pairs):,}")
print(f"Calendar days: {len(dates)}")

# ================================================================
# 6. DEMAND FEATURES
# ================================================================

GROUP_KEYS = ["product_id", "warehouse_id"]

def add_demand_features(frame):
    frame = frame.sort_values(
        GROUP_KEYS + ["date"]
    ).copy()

    grouped = frame.groupby(
        GROUP_KEYS, sort=False
    )["demand"]

    frame["lag_1"] = grouped.shift(1)
    frame["lag_7"] = grouped.shift(7)
    frame["lag_14"] = grouped.shift(14)
    frame["lag_28"] = grouped.shift(28)

    shifted = frame.groupby(
        GROUP_KEYS, sort=False
    )["demand"].shift(1)

    temp = frame[
        GROUP_KEYS + ["date"]
    ].copy()

    temp["shifted_demand"] = shifted

    grouped_shifted = temp.groupby(
        GROUP_KEYS, sort=False
    )["shifted_demand"]

    for window in [7, 14, 28]:
        frame[f"rolling_mean_{window}"] = (
            grouped_shifted.transform(
                lambda x: x.rolling(
                    window, min_periods=window
                ).mean()
            )
        )

        frame[f"rolling_std_{window}"] = (
            grouped_shifted.transform(
                lambda x: x.rolling(
                    window, min_periods=window
                ).std()
            )
        )

    frame["day_of_week"] = (
        frame["date"].dt.dayofweek
    )

    frame["month"] = (
        frame["date"].dt.month
    )

    frame["is_weekend"] = (
        frame["day_of_week"] >= 5
    ).astype(int)

    return frame


panel = add_demand_features(panel)

panel = panel.merge(
    product_supplier,
    on="product_id",
    how="left",
    validate="many_to_one"
)

panel["lead_time_days"] = (
    panel["lead_time_days"].fillna(7)
)

panel["on_time_delivery_rate"] = (
    panel["on_time_delivery_rate"].fillna(0.90)
)

panel["defect_rate"] = (
    panel["defect_rate"].fillna(0.02)
)

# ================================================================
# 7. SIMULATED HISTORICAL INVENTORY LABELS
# ================================================================

# The supplied inventory file is a CURRENT snapshot.
# It does not contain past stock levels.
#
# Therefore, historical stockouts cannot be recovered
# reliably from this dataset.
#
# This section creates a deterministic, simulated
# inventory process for educational model training.
# It does not claim to reconstruct actual stockouts.

inventory_lookup = inventory[
    GROUP_KEYS + [
        "stock_on_hand",
        "reorder_point",
        "safety_stock"
    ]
].copy()

panel = panel.merge(
    inventory_lookup,
    on=GROUP_KEYS,
    how="left",
    validate="many_to_one"
)

panel["stock_on_hand"] = (
    panel["stock_on_hand"].fillna(0)
)

panel["reorder_point"] = (
    panel["reorder_point"].fillna(0)
)

panel["safety_stock"] = (
    panel["safety_stock"].fillna(0)
)

def simulate_group(group):
    group = group.sort_values("date").copy()

    demand_values = group["demand"].to_numpy(float)

    current_stock = float(
        group["stock_on_hand"].iloc[0]
    )

    reorder_point = float(
        group["reorder_point"].iloc[0]
    )

    lead_time = max(
        1,
        int(np.ceil(group["lead_time_days"].iloc[0]))
    )

    # Start with a plausible simulated opening inventory.
    # The current inventory snapshot is not treated as
    # a verified historical opening balance.
    opening_stock = max(
        reorder_point * 1.5,
        float(np.mean(demand_values[:28])) * lead_time * 2,
        1.0
    )

    stock = opening_stock
    pending_receipts = {}

    opening_levels = []
    stockout_labels = []

    for day_idx, demand in enumerate(demand_values):
        stock += pending_receipts.pop(day_idx, 0.0)

        opening_levels.append(stock)

        stockout_labels.append(
            int(demand > stock)
        )

        stock = max(0.0, stock - demand)

        if stock <= reorder_point:
            # Simple simulated fixed order-up-to policy.
            target_stock = max(
                reorder_point * 2,
                np.mean(demand_values[:day_idx + 1])
                * (lead_time + 7),
                1.0
            )

            pending_quantity = sum(
                pending_receipts.values()
            )

            order_quantity = max(
                0.0,
                target_stock - stock - pending_quantity
            )

            if order_quantity > 0:
                arrival = day_idx + lead_time

                pending_receipts[arrival] = (
                    pending_receipts.get(arrival, 0.0)
                    + order_quantity
                )

    group["simulated_opening_stock"] = opening_levels
    group["simulated_stockout"] = stockout_labels

    return group


simulated_parts = []

for _, group in panel.groupby(GROUP_KEYS, sort=False):
    simulated_parts.append(
        simulate_group(group)
    )

panel = pd.concat(
    simulated_parts,
    ignore_index=True
)

panel["simulated_inventory_cover"] = (
    panel["simulated_opening_stock"]
    /
    panel["rolling_mean_14"].clip(lower=0.1)
)

# ================================================================
# 8. MODEL TRAINING DATA
# ================================================================

FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28",
    "day_of_week",
    "month",
    "is_weekend",
    "lead_time_days",
    "on_time_delivery_rate",
    "defect_rate",
    "simulated_opening_stock",
    "simulated_inventory_cover"
]

model_data = panel.dropna(
    subset=FEATURES
).copy()

test_start = (
    model_data["date"].max()
    - pd.Timedelta(days=TEST_DAYS - 1)
)

train_data = model_data[
    model_data["date"] < test_start
].copy()

test_data = model_data[
    model_data["date"] >= test_start
].copy()

if train_data.empty or test_data.empty:
    raise ValueError(
        "Insufficient historical data for training/test."
    )

X_train = train_data[FEATURES]
y_train = train_data["simulated_stockout"].astype(int)

X_test = test_data[FEATURES]
y_test = test_data["simulated_stockout"].astype(int)

print("\nTRAIN / TEST")
print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print(f"Training positive rate: {y_train.mean():.2%}")
print(f"Test positive rate: {y_test.mean():.2%}")

# ================================================================
# 9. TRAIN CLASSIFIER
# ================================================================

model = None
test_probability = None

if y_train.nunique() >= 2:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    positive_class_index = list(
        model.classes_
    ).index(1)

    test_probability = model.predict_proba(
        X_test
    )[:, positive_class_index]

    test_predicted = (
        test_probability >= RISK_THRESHOLD
    ).astype(int)

    print("\nCLASSIFICATION REPORT")
    print(
        classification_report(
            y_test,
            test_predicted,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_test,
        test_predicted,
        labels=[0, 1]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "No Stockout",
            "Stockout"
        ]
    ).plot(cmap="Blues")

    plt.title("Simulated Stockout Classification")
    plt.tight_layout()
    plt.savefig(
        REPORT_DIR / "confusion_matrix.png"
    )
    plt.show()

    test_metrics = {
        "roc_auc": (
            float(roc_auc_score(
                y_test, test_probability
            ))
            if y_test.nunique() == 2
            else None
        ),
        "average_precision": float(
            average_precision_score(
                y_test, test_probability
            )
        ),
        "brier_score": float(
            brier_score_loss(
                y_test, test_probability
            )
        )
    }

else:
    print(
        "\nOnly one simulated class found. "
        "Skipping classifier training."
    )

    test_metrics = {
        "roc_auc": None,
        "average_precision": None,
        "brier_score": None
    }

print("\nTEST METRICS")
print(json.dumps(test_metrics, indent=2))

# ================================================================
# 10. CURRENT INVENTORY RISK FEATURES
# ================================================================

latest_features = (
    panel.sort_values("date")
    .groupby(GROUP_KEYS, as_index=False)
    .tail(1)
    .copy()
)

latest_features = latest_features.merge(
    inventory_lookup,
    on=GROUP_KEYS,
    how="left",
    suffixes=("", "_current")
)

# Current inventory comes from the real snapshot,
# not the simulated historical stock trajectory.
latest_features["current_stock"] = (
    latest_features["stock_on_hand"]
    .fillna(0)
    .clip(lower=0)
)

latest_features["avg_daily_demand"] = (
    latest_features["rolling_mean_28"]
    .fillna(latest_features["rolling_mean_14"])
    .fillna(latest_features["rolling_mean_7"])
    .fillna(0)
    .clip(lower=0)
)

latest_features["daily_demand_std"] = (
    latest_features["rolling_std_28"]
    .fillna(latest_features["rolling_std_14"])
    .fillna(latest_features["rolling_std_7"])
    .fillna(0)
    .clip(lower=0)
)

latest_features["lead_time_demand"] = (
    latest_features["avg_daily_demand"]
    * latest_features["lead_time_days"]
)

latest_features["recommended_safety_stock"] = (
    SERVICE_LEVEL_Z
    * latest_features["daily_demand_std"]
    * np.sqrt(
        latest_features["lead_time_days"].clip(lower=1)
    )
)

latest_features["recommended_reorder_point"] = (
    latest_features["lead_time_demand"]
    + latest_features["recommended_safety_stock"]
)

latest_features["days_of_cover"] = np.where(
    latest_features["avg_daily_demand"] > 0,
    latest_features["current_stock"]
    / latest_features["avg_daily_demand"].replace(0, np.nan),
    np.nan
)

# ================================================================
# 11. STOCKOUT RISK ESTIMATION
# ================================================================

# Probabilistic risk from a normal approximation to
# demand over lead time. This is a heuristic, not a
# calibrated empirical stockout probability.
#
# P(Demand during lead time > available stock)
#
# Approximate mean = daily mean * lead time
# Approximate std  = daily std * sqrt(lead time)

from math import erf, sqrt

def normal_survival(z):
    return 0.5 * (1.0 - erf(z / sqrt(2.0)))


def estimate_risk(row):
    stock = float(row["current_stock"])
    mean = float(row["lead_time_demand"])
    std = float(row["daily_demand_std"]) * np.sqrt(
        max(float(row["lead_time_days"]), 1)
    )

    if std <= 1e-8:
        return float(mean > stock)

    z = (stock - mean) / std

    return float(np.clip(
        normal_survival(z),
        0,
        1
    ))


latest_features["stockout_risk_score"] = (
    latest_features.apply(
        estimate_risk,
        axis=1
    )
)

latest_features["risk_category"] = pd.cut(
    latest_features["stockout_risk_score"],
    bins=[-0.001, 0.25, 0.50, 0.75, 1.001],
    labels=[
        "Low",
        "Medium",
        "High",
        "Critical"
    ],
    right=False
)

# The classifier was trained on simulated inventory.
# Its output is provided separately and must not be
# presented as a validated real-world probability.

if model is not None:
    current_ml_features = latest_features[
        FEATURES
    ].copy()

    current_ml_features["simulated_opening_stock"] = (
        latest_features["current_stock"]
    )

    current_ml_features["simulated_inventory_cover"] = (
        latest_features["days_of_cover"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(999)
    )

    current_ml_features = current_ml_features.fillna(0)

    latest_features["simulated_ml_score"] = (
        model.predict_proba(
            current_ml_features
        )[:, positive_class_index]
    )
else:
    latest_features["simulated_ml_score"] = np.nan

# ================================================================
# 12. REPLENISHMENT RECOMMENDATIONS
# ================================================================

latest_features["target_stock"] = (
    latest_features["avg_daily_demand"]
    * (
        latest_features["lead_time_days"]
        + 7
    )
    + latest_features["recommended_safety_stock"]
)

latest_features["recommended_order_qty"] = np.ceil(
    (
        latest_features["target_stock"]
        - latest_features["current_stock"]
    ).clip(lower=0)
).astype(int)

latest_features["reorder_required"] = (
    latest_features["current_stock"]
    <= latest_features["recommended_reorder_point"]
)

latest_features["priority"] = np.select(
    [
        latest_features["stockout_risk_score"] >= 0.75,
        latest_features["stockout_risk_score"] >= 0.50,
        latest_features["reorder_required"]
    ],
    [
        "P1 - Critical",
        "P2 - High",
        "P3 - Reorder"
    ],
    default="P4 - Monitor"
)

latest_features["recommendation"] = np.select(
    [
        latest_features["priority"] == "P1 - Critical",
        latest_features["priority"] == "P2 - High",
        latest_features["reorder_required"]
    ],
    [
        "Review urgently; expedite replenishment",
        "Create replenishment plan",
        "Review reorder quantity"
    ],
    default="Monitor inventory"
)

# ================================================================
# 13. FINAL RISK TABLE
# ================================================================

OUTPUT_COLUMNS = [
    "product_id",
    "warehouse_id",
    "supplier_id",
    "current_stock",
    "avg_daily_demand",
    "daily_demand_std",
    "lead_time_days",
    "lead_time_demand",
    "days_of_cover",
    "recommended_safety_stock",
    "recommended_reorder_point",
    "stockout_risk_score",
    "simulated_ml_score",
    "risk_category",
    "recommended_order_qty",
    "reorder_required",
    "priority",
    "recommendation"
]

risk_results = latest_features[
    OUTPUT_COLUMNS
].copy()

risk_results = risk_results.sort_values(
    "stockout_risk_score",
    ascending=False
)

print("\nTOP 20 STOCKOUT-RISK SKU/WAREHOUSE PAIRS")
print(
    risk_results.head(20).round(3).to_string(index=False)
)

# ================================================================
# 14. RISK DISTRIBUTION CHART
# ================================================================

risk_counts = (
    risk_results["risk_category"]
    .value_counts()
    .reindex(
        ["Low", "Medium", "High", "Critical"],
        fill_value=0
    )
)

plt.figure(figsize=(10, 5))

plt.bar(
    risk_counts.index.astype(str),
    risk_counts.values
)

plt.title("Current Stockout Risk Distribution")
plt.xlabel("Risk Category")
plt.ylabel("SKU / Warehouse Count")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "risk_distribution.png"
)
plt.show()

# ================================================================
# 15. TOP STOCKOUT-RISK PRODUCTS
# ================================================================

top_risk = risk_results.head(20).copy()

top_risk["sku_location"] = (
    top_risk["product_id"].astype(str)
    + " / "
    + top_risk["warehouse_id"].astype(str)
)

plt.figure(figsize=(12, 8))

plt.barh(
    top_risk["sku_location"].iloc[::-1],
    top_risk["stockout_risk_score"].iloc[::-1]
)

plt.xlim(0, 1)
plt.xlabel("Estimated Risk Score")
plt.title("Top 20 SKU / Warehouse Stockout Risks")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "top_stockout_risks.png"
)
plt.show()

# ================================================================
# 16. INVENTORY COVERAGE CHART
# ================================================================

coverage_data = risk_results[
    np.isfinite(risk_results["days_of_cover"])
].copy()

plt.figure(figsize=(11, 5))

plt.hist(
    coverage_data["days_of_cover"].clip(upper=90),
    bins=25
)

plt.xlabel("Days of Inventory Cover")
plt.ylabel("SKU / Warehouse Count")
plt.title("Inventory Coverage Distribution")
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "inventory_coverage.png"
)
plt.show()

# ================================================================
# 17. FEATURE IMPORTANCE
# ================================================================

if model is not None:
    feature_importance = pd.DataFrame({
        "feature": FEATURES,
        "importance": model.feature_importances_
    }).sort_values(
        "importance",
        ascending=False
    )

    plt.figure(figsize=(11, 7))

    plt.barh(
        feature_importance["feature"].iloc[::-1],
        feature_importance["importance"].iloc[::-1]
    )

    plt.title(
        "Random Forest Feature Importance "
        "(Simulated Labels)"
    )
    plt.tight_layout()

    plt.savefig(
        REPORT_DIR / "feature_importance.png"
    )
    plt.show()

    feature_importance.to_csv(
        REPORT_DIR / "feature_importance.csv",
        index=False
    )

# ================================================================
# 18. OPTIONAL AGGREGATE FORECAST COMPARISON
# ================================================================

if aggregate_forecast is not None:
    if "predicted_demand" in aggregate_forecast.columns:
        forecast_total = pd.to_numeric(
            aggregate_forecast["predicted_demand"],
            errors="coerce"
        ).sum()

        historical_daily_total = (
            sales_daily.groupby("date")["demand"]
            .sum()
            .tail(30)
            .mean()
        )

        print("\nAGGREGATE FORECAST REFERENCE")
        print(f"Forecast total units: {forecast_total:,.2f}")
        print(
            f"Historical average daily units: "
            f"{historical_daily_total:,.2f}"
        )
        print(
            "Aggregate forecasts are not directly assigned "
            "to individual SKUs in this notebook."
        )

# ================================================================
# 19. BUSINESS KPIs
# ================================================================

critical_count = int(
    (risk_results["risk_category"] == "Critical").sum()
)

high_count = int(
    (risk_results["risk_category"] == "High").sum()
)

reorder_count = int(
    risk_results["reorder_required"].sum()
)

recommended_units = int(
    risk_results["recommended_order_qty"].sum()
)

kpis = {
    "sku_warehouse_pairs": int(len(risk_results)),
    "critical_risk_pairs": critical_count,
    "high_risk_pairs": high_count,
    "reorder_required_pairs": reorder_count,
    "recommended_order_units": recommended_units,
    "average_risk_score": float(
        risk_results["stockout_risk_score"].mean()
    )
}

print("\nSTOCKOUT PREDICTION KPIs")
print(json.dumps(kpis, indent=2))

# ================================================================
# 20. EXPORT RESULTS
# ================================================================

risk_results.to_csv(
    PROCESSED_DIR / "stockout_predictions.csv",
    index=False
)

risk_results[
    risk_results["reorder_required"]
].to_csv(
    PROCESSED_DIR / "replenishment_recommendations.csv",
    index=False
)

model_data[
    GROUP_KEYS
    + ["date", "simulated_stockout"]
    + FEATURES
].to_csv(
    REPORT_DIR / "simulated_training_data.csv",
    index=False
)

if model is not None:
    model_artifact = {
        "model": model,
        "features": FEATURES,
        "threshold": RISK_THRESHOLD,
        "training_label": "simulated_stockout",
        "is_real_world_validated": False,
        "training_end_date": str(
            train_data["date"].max().date()
        ),
        "test_metrics": test_metrics
    }

    joblib.dump(
        model_artifact,
        MODEL_DIR / "stockout_classifier.joblib"
    )

report = {
    "project": "SupplyChainAI",
    "notebook": "03_stockout_prediction.ipynb",
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "label_source": "simulated_inventory_trajectory",
    "real_world_validated": False,
    "risk_method": (
        "Normal approximation to lead-time demand"
    ),
    "classification_model": (
        "RandomForestClassifier"
        if model is not None
        else None
    ),
    "test_metrics_on_simulated_labels": test_metrics,
    "kpis": kpis,
    "limitations": [
        "Current inventory is a snapshot, not a history.",
        "Historical stockout labels are simulated.",
        "Observed sales may underestimate true demand.",
        "Risk scores are not empirically calibrated probabilities.",
        "Lead times are treated as fixed.",
        "Open purchase orders and inbound inventory are not modeled.",
        "Reorder quantities ignore minimum order quantities and budgets.",
        "No automatic purchase orders are submitted."
    ]
}

with open(
    REPORT_DIR / "stockout_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(report, file, indent=2)

# ================================================================
# 21. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 75)
print("SUPPLYCHAINAI — STOCKOUT ANALYSIS COMPLETED")
print("=" * 75)

print(f"SKU/warehouse pairs: {len(risk_results):,}")
print(f"Critical risk: {critical_count}")
print(f"High risk: {high_count}")
print(f"Reorder required: {reorder_count}")
print(f"Suggested order units: {recommended_units:,}")

print("\nGenerated files:")
print("data/processed/stockout_predictions.csv")
print("data/processed/replenishment_recommendations.csv")
print("reports/stockout_prediction/stockout_report.json")
print("reports/stockout_prediction/risk_distribution.png")
print("reports/stockout_prediction/top_stockout_risks.png")
print("reports/stockout_prediction/inventory_coverage.png")

if model is not None:
    print("models/stockout_prediction/stockout_classifier.joblib")

print("\nRecommended next notebooks:")
print("04_inventory_optimization.ipynb")
print("05_supplier_risk_model.ipynb")
print("06_logistics_optimization.ipynb")

print("=" * 75)
